In [ ]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [ ]:
!pip install transformers==4.41.2 datasets==2.19.1 accelerate==0.30.1 peft==0.11.1

In [ ]:
!pip install transformers[sentencepiece] datasets sacrebleu rouge_score py7zr -q

In [ ]:
from collections import UserString
from transformers import pipeline, set_seed
from datasets import load_dataset, load_from_disk
import matplotlib.pyplot as plt
import pandas as pd
from datasets import load_dataset, load_metric

from transformers import AutoModelForSeq2SeqLM #for loading a transformer based model which we will be UserString
from transformers import AutoTokenizer

import nltk
from nltk.tokenize import sent_tokenize
from tqdm import tqdm
import torch

nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


preprocessing our data and converting text to vectors

In [ ]:
model_checkpoint = "google/pegasus-cnn_dailymail"
#LOADING TOKENIZER
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


LOADING MODEL

In [ ]:
model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint).to(device) #loading the model to device which is gpu as gpu is currently working on this session

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-cnn_dailymail and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
dataset = load_dataset("knkarthick/samsum")

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
})

In [ ]:
dataset['train'][0]

{'id': '13818513',
 'dialogue': "Amanda: I baked  cookies. Do you want some?\nJerry: Sure!\nAmanda: I'll bring you tomorrow :-)",
 'summary': 'Amanda baked cookies and will bring Jerry some tomorrow.'}

Creating function for trokenization

In [ ]:
def convert_examples_to_features(example_batch):
  input_encodings = tokenizer(example_batch['dialogue'], max_length=1024, truncation=True)

  with tokenizer.as_target_tokenizer():
    target_encodings = tokenizer(example_batch['summary'], max_length=128, truncation=True)

  return{
      'input_ids': input_encodings['input_ids'],
      'attention_mask': input_encodings['attention_mask'],
      'labels' : target_encodings['input_ids']
  }


In [ ]:
tokenized_dataset = dataset.map(convert_examples_to_features, batched=True) #argument ensures that the processing is done in batches, which is more efficient.


Map:   0%|          | 0/818 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:3946: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


In [ ]:
tokenized_dataset['train'][1]

{'id': '13728867',
 'dialogue': 'Olivia: Who are you voting for in this election? \nOliver: Liberals as always.\nOlivia: Me too!!\nOliver: Great',
 'summary': 'Olivia and Olivier are voting for liberals in this election. ',
 'input_ids': [18038,
  151,
  2632,
  127,
  119,
  6228,
  118,
  115,
  136,
  2974,
  152,
  10463,
  151,
  35884,
  130,
  329,
  107,
  18038,
  151,
  2587,
  314,
  1242,
  10463,
  151,
  1509,
  1],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 'labels': [18038, 111, 34296, 127, 6228, 118, 33195, 115, 136, 2974, 107, 1]}

since we have low memory we can load data in batches

In [ ]:
from transformers import DataCollatorForSeq2Seq
seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model = model_pegasus)

creatin training args

In [ ]:
pip install transformers==4.44.2 accelerate==0.34.2 torch==2.1.2

  Using cached transformers-4.44.2-py3-none-any.whl.metadata (43 kB)
  Using cached accelerate-0.34.2-py3-none-any.whl.metadata (19 kB)
ERROR: Could not find a version that satisfies the requirement torch==2.1.2 (from versions: 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.3.1, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2.8.0, 2.9.0, 2.9.1, 2.10.0, 2.11.0)
ERROR: No matching distribution found for torch==2.1.2


In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir='pegasus-samsum', # Directory where model checkpoints and outputs will be saved.
    num_train_epochs=1,          # Number of training epochs. An epoch is one full pass over the training data.
    warmup_steps = 500,          # Number of steps for the learning rate to linearly warm up from 0 to its peak value.
    per_device_train_batch_size=1, # Batch size per GPU/TPU core/CPU for training.
    per_device_eval_batch_size=1,  # Batch size per GPU/TPU core/CPU for evaluation.
    weight_decay=0.01,           # Strength of weight decay (L2 regularization) to prevent overfitting.
    logging_steps=10,            # Number of update steps between two loggings. Metrics like loss will be reported every 10 steps.
    eval_strategy='steps',       # Evaluation is performed at specified intervals based on 'steps' rather than 'epoch' boundaries. (Updated from evaluation_strategy)
                                 #   - 'steps': Evaluation occurs every `eval_steps`.
                                 #   - 'epoch': Evaluation occurs at the end of each epoch.
                                 # Choosing 'steps' allows for more granular and frequent monitoring of training progress, especially
                                 # for long epochs or when you want to evaluate before a full epoch completes.
    eval_steps=500,              # Number of update steps between two evaluations. The model will be evaluated every 500 training steps.
    save_steps=1e6,              # Number of update steps between two checkpoints. A large value like 1e6 means saving very infrequently.
    gradient_accumulation_steps=32, # Number of updates steps to accumulate the gradients for, before performing a backward/update pass.
                                 # This effectively simulates a larger batch size (per_device_train_batch_size * gradient_accumulation_steps)
                                 # without requiring more GPU memory for each individual step.
                                 # Here, an effective batch size of 1 * 32 = 32 is simulated.
    use_cpu=False,               # Set to False to enable GPU usage.
    report_to='none'             # Disable reporting to experiment tracking platforms like Weights & Biases.
)

Initisalizing trainer

In [ ]:
trainer = Trainer(model = model_pegasus, args = training_args,
                  tokenizer = tokenizer,
                  data_collator = seq2seq_data_collator,
                  train_dataset = tokenized_dataset['test'],  #using test data as train for experimenting as train data is huge and will takje time
                  eval_dataset = tokenized_dataset['validation'])


Training

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss


TrainOutput(global_step=25, training_loss=3.0824682235717775, metrics={'train_runtime': 209.662, 'train_samples_per_second': 3.906, 'train_steps_per_second': 0.119, 'total_flos': 307474003279872.0, 'train_loss': 3.0824682235717775, 'epoch': 0.9768009768009768})

In [ ]:
rouge_names = ['rouge1','rouge2','rougeL','rougeLsum']
rouge_metrics = load_metrics('rouge')

score  = calculate_metric_on_test_ds(
    dataset_samsum['test'][0:10], rouge_metrics, trainer.model, tokenizer, batch_size = 2, column_text = 'dialogue', column_summary = 'summary'
)
rouge_dict = dict((rn, score[rn].mid.fmeasure) for rn in rouge_names)

pd.DataFrame(rouge_dict, index = [f'pegasus'])

#rouge score close to 1 means that the model is performing good

In [ ]:
# Prediction settings
gen_kwargs = {"length_penalty": 0.8, "num_beams": 8, "max_length": 128}

# Sample input
sample_text = dataset_samsum["test"][0]["dialogue"]
reference = dataset_samsum["test"][0]["summary"]

# Create pipeline
pipe = pipeline("summarization", model=model, tokenizer=tokenizer)

# Generate summary
prediction = pipe(sample_text, **gen_kwargs)[0]["summary_text"]

# Print results
print("Dialogue:")
print(sample_text)

print("\nReference Summary:")
print(reference)

print("\nModel Summary:")
print(prediction)
What was missing in your snip

saving model

In [ ]:
model_pegasus.save_pretrained('pegasus-samsum-model')

saving tokenizer

In [ ]:
tokenizer.save_pretrained('tokenizer')

('tokenizer/tokenizer_config.json',
 'tokenizer/special_tokens_map.json',
 'tokenizer/spiece.model',
 'tokenizer/added_tokens.json',
 'tokenizer/tokenizer.json')